<a href="https://colab.research.google.com/github/abhigaur2006-commits/sales-revenue-analysis-dashboard/blob/main/Sales_Revenue_Analysis_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
df = pd.read_csv("train.csv")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
0,1,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600
1,2,CA-2017-152156,08/11/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400
2,3,CA-2017-138688,12/06/2017,16/06/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200
3,4,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775
4,5,US-2016-108966,11/10/2016,18/10/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680


In [10]:
print(df.shape)
print(df.columns.tolist())
df.info()

(9800, 18)
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   object 
 2   Order Date     9800 non-null   object 
 3   Ship Date      9800 non-null   object 
 4   Ship Mode      9800 non-null   object 
 5   Customer ID    9800 non-null   object 
 6   Customer Name  9800 non-null   object 
 7   Segment        9800 non-null   object 
 8   Country        9800 non-null   object 
 9   City           9800 non-null   object 
 10  State          9800 non-null   object 
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   obje

In [17]:
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True)
df = df.drop_duplicates()
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.to_period("M").astype(str)
df[["Order Date", "Year", "Month"]].head()

,Order Date,Year,Month
0,2017-11-08,2017,2017-11
1,2017-11-08,2017,2017-11
2,2017-06-12,2017,2017-06
3,2016-10-11,2016,2016-10
4,2016-10-11,2016,2016-10


In [12]:
total_sales = df["Sales"].sum()
total_orders = df["Order ID"].nunique()
avg_order_value = total_sales / total_orders
top_product = df.groupby("Product Name")["Sales"].sum().idxmax()

print(f"Total Sales: ${total_sales:,.0f}")
print(f"Total Orders: {total_orders:,}")
print(f"Avg Order Value: ${avg_order_value:,.2f}")
print(f"Top Product: {top_product}")

Total Sales: $2,261,537
Total Orders: 4,922
Avg Order Value: $459.48
Top Product: Canon imageCLASS 2200 Advanced Copier


Monthly trend: Sales grew from $480K in 2015 to $722K in 2018, with a small dip in 2016. Sales peak every Nov/Dec: these two months make up about 30% of all sales, and the highest month was Nov 2018 (about $118K).


In [13]:
import plotly.express as px

monthly = df.groupby("Month")["Sales"].sum().reset_index()
px.line(monthly, x="Month", y="Sales", title="Monthly Sales Trend")

Top products and regions: The Canon imageCLASS 2200 Advanced Copier is the top product by a wide margin (about $61.6K), and the top 10 is mostly office technology such as copiers, binding machines, and printers. West ($710K) and East ($670K) together bring in about 61% of total sales, while South is the lowest ($389K).


In [14]:
top10 = (df.groupby("Product Name")["Sales"].sum()
           .nlargest(10).sort_values().reset_index())
px.bar(top10, x="Sales", y="Product Name", orientation="h",
       title="Top 10 Products by Sales")

Conclusion: The business is growing, with 2018 sales about 20% higher than 2017. It should stock up and run promotions before November and December, keep strong inventory of its top products, and put more marketing effort into the South region, which is the weakest. Total sales were $2.26M across 4,922 orders, with an average order value of about $459.

In [15]:
reg = df.groupby("Region")["Sales"].sum().reset_index()
px.bar(reg, x="Region", y="Sales", title="Sales by Region", color="Region")

In [21]:
import ipywidgets as widgets

def dashboard(Region, Category, Year):
    d = df.copy()
    if Region != "All":
        d = d[d["Region"] == Region]
    if Category != "All":
        d = d[d["Category"] == Category]
    if Year != "All":
        d = d[d["Year"] == Year]

    print(f"Total Sales: ${d['Sales'].sum():,.0f}")
    print(f"Orders: {d['Order ID'].nunique():,}")
    print(f"Avg Order Value: ${d['Sales'].sum() / max(d['Order ID'].nunique(), 1):,.2f}")

    m = d.groupby("Month")["Sales"].sum().reset_index()
    px.line(m, x="Month", y="Sales", title="Monthly Sales (filtered)").show()

    t = d.groupby("Sub-Category")["Sales"].sum().nlargest(10).sort_values().reset_index()
    px.bar(t, x="Sales", y="Sub-Category", orientation="h",
           title="Top Sub-Categories (filtered)").show()

widgets.interact(
    dashboard,
    Region=["All"] + sorted(df["Region"].unique().tolist()),
    Category=["All"] + sorted(df["Category"].unique().tolist()),
    Year=["All"] + sorted(df["Year"].unique().tolist()),
);

interactive(children=(Dropdown(description='Region', options=('All', 'Central', 'East', 'South', 'West'), valu…